In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch
import torch.optim as optim
import torch.nn as nn
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/zalando-research/fashionmnist/t10k-labels-idx1-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/t10k-images-idx3-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_test.csv
/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_train.csv
/kaggle/input/datasets/zalando-research/fashionmnist/train-labels-idx1-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/train-images-idx3-ubyte


In [2]:
df=pd.read_csv("/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_train.csv")

In [3]:
X=df.iloc[:, 1:].values

In [4]:
Y=df.iloc[:, 0].values

In [5]:
x_train, x_test , y_train,y_test = train_test_split(X,Y,test_size=0.2, random_state=42)

In [6]:
x_train= x_train/255.0
x_test = x_test/255.0

In [7]:
class CustomDataset(Dataset):
    
    def __init__(self,features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels,dtype=torch.long)


    def __len__(self):
        return len(self.features)

    def __getitem__(self,index):
        return self.features[index], self.labels[index]




   

In [8]:
train_dataset = CustomDataset(x_train,y_train)
test_dataset = CustomDataset(x_test,y_test)

In [9]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [10]:
class NN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.model= nn.Sequential(
            nn.Linear(num_features,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,16),
            nn.ReLU(),
            nn.Linear(16,10)
        )
    def forward(self,x):
            return self.model(x)

In [11]:
epochs=100
learning_rate=0.1

In [12]:
model = NN(x_train.shape[1])

In [13]:
criteria= nn.CrossEntropyLoss()

In [14]:
optimizer=optim.SGD(model.parameters(),lr=learning_rate)

In [15]:
for epoch in range(epochs):
    epoch_loss=0
    for batch_features, batch_labels in train_loader:
        output = model(batch_features)
        loss = criteria(output,batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss=epoch_loss+loss.item()

    avg_loss= epoch_loss/len(train_loader)
    print(avg_loss)

0.7631154541373253
0.45369328880310056
0.4043943038781484
0.37056561809778216
0.3478028996313612
0.3283974430610736
0.3144737497890989
0.3035653274382154
0.29096318284794687
0.2834878947635492
0.2737236564370493
0.26538329347719747
0.2586444317338367
0.2514088877737522
0.2476677193554739
0.24103846840312085
0.23388498509923616
0.22774669223154584
0.22476720432937145
0.22089646720016995
0.2164041155365606
0.21343492891763646
0.20696135710179805
0.2050954170562327
0.20161031023288767
0.1975700546031197
0.1924648302681744
0.1906440656327953
0.1876917657467226
0.18441756898040573
0.18277403031770761
0.18624530207862455
0.17470781140960753
0.17344066019852955
0.1714610721981153
0.16687307233021906
0.1626561724053075
0.16799078647233545
0.1613399874943619
0.15949007906299084
0.15338237358598658
0.15922230708909532
0.15052665037421198
0.144243390379784
0.14941999361974498
0.1453950203753387
0.14209086691650252
0.13993634617701173
0.13884367915491264
0.13720930410184276
0.13439047397964168
0.1

In [16]:
model.eval()

NN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=10, bias=True)
  )
)

In [17]:
def check_accuracy(model, test_loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for features, labels in test_loader:

            output = model(features)

            predictions = torch.argmax(output, dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total

    return accuracy

In [18]:
acc= check_accuracy(model,test_loader)

In [19]:
print(acc)

0.8870833333333333
